# Notebook 04: The Quantum Minority Game

**QCCCM** — Quantum Compute for Computational Cognitive Modeling

---

## What you'll learn

1. The **Minority Game** as a model of bounded rationality and anti-coordination
2. How classical agents self-organize (or fail to) through strategy adaptation
3. The **phase transition** at α_c ≈ 0.34 separating coordination from herding
4. How **quantum coherence** in strategy selection changes collective dynamics
5. Whether quantum agents can break the herding trap

### Background

The Minority Game (Challet & Zhang, 1997) is one of the simplest models of collective decision-making under competition. N agents simultaneously choose one of two options. The **minority** wins — those on the less popular side get rewarded. This captures the essence of:

- **Bar attendance** (El Farol problem): you want to go when it's not crowded
- **Financial markets**: you profit by being contrarian
- **Resource allocation**: anti-coordination is socially optimal

The twist: agents adapt using strategies based on shared history, leading to emergent phenomena including a phase transition between efficient and inefficient regimes.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from qcccm.games.minority import (
    MinorityGameParams,
    run_minority_game,
    volatility_sweep,
)

plt.rcParams['figure.dpi'] = 120
print("Minority Game module loaded")

## 1. The Rules

- **N agents** (odd number) each choose 0 or 1 simultaneously
- The **minority side** wins: if 40 choose 0 and 61 choose 1, those who chose 0 win
- Each agent has **M memory**: they see the last M winning sides as a binary string
- Each agent has **S strategies**: lookup tables mapping each possible history pattern to an action
- After each round, agents score their strategies: +1 if it predicted the winner, −1 if not
- Each agent plays their **best-scoring strategy**

The key control parameter is **α = 2^M / N** — the ratio of possible history patterns to agents.

In [ ]:
# Run a basic classical minority game
params = MinorityGameParams(n_agents=101, memory=3, n_rounds=500, seed=42)
result = run_minority_game(params, quantumness=0.0)

fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)

# Attendance time series
axes[0].plot(result.attendance, 'b-', alpha=0.6, linewidth=0.5)
axes[0].axhline(y=params.n_agents / 2, color='red', linestyle='--', label='N/2', alpha=0.7)
axes[0].set_ylabel('Attendance (# choosing 1)')
axes[0].set_title(f'Minority Game: N={params.n_agents}, M={params.memory}, α={2**params.memory/params.n_agents:.3f}')
axes[0].legend()

# Running volatility
window = 50
running_var = [np.var(result.attendance[max(0,i-window):i+1]) / params.n_agents
               for i in range(len(result.attendance))]
axes[1].plot(running_var, 'g-', linewidth=1)
axes[1].axhline(y=0.25, color='red', linestyle='--', label='Random (σ²/N = 0.25)', alpha=0.7)
axes[1].set_ylabel('σ²/N (volatility)')
axes[1].set_xlabel('Round')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"Overall volatility σ²/N = {result.volatility:.4f}")
print(f"Random benchmark: 0.25")
print(f"{'Better' if result.volatility < 0.25 else 'Worse'} than random!")

## 2. The Phase Transition

The collective behavior depends critically on α = 2^M / N:

- **α < α_c ≈ 0.34** (crowded phase): Too many agents compete over too few history patterns. Strategies become correlated, agents herd → σ²/N > 0.25 (worse than random).
- **α > α_c** (uncrowded phase): Enough history patterns for agents to diversify. Strategies decorrelate → σ²/N < 0.25 (better than random).
- **α → ∞**: Agents have too few samples to learn → approach random (σ²/N → 0.25).

This phase transition is the hallmark of the minority game and connects to frustration in spin glasses.

In [ ]:
# Phase diagram: classical agents
alpha_c, vol_c_mean, vol_c_std = volatility_sweep(
    n_agents=101, memory_range=range(1, 9),
    n_rounds=500, n_seeds=5, quantumness=0.0,
)

fig, ax = plt.subplots(figsize=(9, 6))
ax.errorbar(alpha_c, vol_c_mean, yerr=vol_c_std, fmt='bo-', linewidth=2,
            markersize=6, capsize=4, label='Classical agents')
ax.axhline(y=0.25, color='red', linestyle='--', linewidth=1.5, label='Random benchmark')
ax.set_xscale('log')
ax.set_xlabel('α = 2^M / N')
ax.set_ylabel('σ²/N')
ax.set_title('Minority Game Phase Diagram (N=101)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("α values:", [f"{a:.3f}" for a in alpha_c])
print("σ²/N:   ", [f"{v:.3f}" for v in vol_c_mean])

## 3. Quantum Agents: Breaking the Herding Trap

Classical agents in the crowded phase (low α) all flock to the same "best" strategy simultaneously, causing herding and high volatility. This is because strategy selection is a hard switch — the best strategy wins completely.

**Quantum agents** maintain a density matrix over strategies instead. The `quantumness` parameter q controls off-diagonal coherences:
- q = 0: hard switch (classical)
- q > 0: superposition of strategies with interference

The interference can **decorrelate** agents even in the crowded phase, because:
1. Different agents accumulate different phases in their strategy superpositions
2. Measurement (strategy selection) projects onto different strategies for different agents
3. This effectively adds "strategic noise" that is *structured* — not random, but interference-driven

In [ ]:
# Compare classical vs quantum across the phase diagram
quantumness_values = [0.0, 0.2, 0.5]

fig, ax = plt.subplots(figsize=(9, 6))

colors = ['blue', 'orange', 'green']
for q, color in zip(quantumness_values, colors):
    alpha, vol_mean, vol_std = volatility_sweep(
        n_agents=101, memory_range=range(1, 9),
        n_rounds=500, n_seeds=5, quantumness=q,
    )
    label = 'Classical' if q == 0 else f'Quantum (q={q})'
    ax.errorbar(alpha, vol_mean, yerr=vol_std, fmt='o-', color=color,
                linewidth=2, markersize=6, capsize=4, label=label)

ax.axhline(y=0.25, color='red', linestyle='--', linewidth=1.5, label='Random', alpha=0.7)
ax.set_xscale('log')
ax.set_xlabel('α = 2^M / N')
ax.set_ylabel('σ²/N')
ax.set_title('Quantum vs Classical Minority Game Phase Diagram')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Focus on the crowded phase (low α) where herding is worst
# Compare volatility at M=2 (α ≈ 0.04) across quantumness values
q_values = np.linspace(0, 0.8, 12)
vols = []

for q in q_values:
    seeds_vols = []
    for seed in range(10):
        params = MinorityGameParams(n_agents=101, memory=2, n_rounds=500, seed=seed)
        result = run_minority_game(params, quantumness=q)
        seeds_vols.append(result.volatility)
    vols.append((np.mean(seeds_vols), np.std(seeds_vols)))

vol_means = [v[0] for v in vols]
vol_stds = [v[1] for v in vols]

fig, ax = plt.subplots(figsize=(8, 5))
ax.errorbar(q_values, vol_means, yerr=vol_stds, fmt='ko-', linewidth=2,
            markersize=6, capsize=4)
ax.axhline(y=0.25, color='red', linestyle='--', alpha=0.7, label='Random benchmark')
ax.axhline(y=vol_means[0], color='blue', linestyle=':', alpha=0.7, label=f'Classical σ²/N = {vol_means[0]:.3f}')
ax.set_xlabel('Quantumness (q)')
ax.set_ylabel('σ²/N')
ax.set_title('Effect of Quantumness in the Crowded Phase (M=2, α≈0.04)')
ax.legend()
plt.tight_layout()
plt.show()

print(f"Classical volatility (q=0): {vol_means[0]:.4f}")
best_q = q_values[np.argmin(vol_means)]
print(f"Best quantum volatility: {min(vol_means):.4f} at q={best_q:.2f}")

## 4. Attendance Dynamics: Classical vs Quantum

Let's look at the time series side by side to visualise how quantum coherence changes the attendance dynamics.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)

params = MinorityGameParams(n_agents=101, memory=2, n_rounds=300, seed=42)

# Classical
result_c = run_minority_game(params, quantumness=0.0)
axes[0].plot(result_c.attendance, 'b-', alpha=0.7, linewidth=0.6)
axes[0].axhline(y=50.5, color='red', linestyle='--', alpha=0.5)
axes[0].set_ylabel('Attendance')
axes[0].set_title(f'Classical (q=0): σ²/N = {result_c.volatility:.3f}')
axes[0].set_ylim(20, 80)

# Quantum
result_q = run_minority_game(params, quantumness=0.4)
axes[1].plot(result_q.attendance, 'g-', alpha=0.7, linewidth=0.6)
axes[1].axhline(y=50.5, color='red', linestyle='--', alpha=0.5)
axes[1].set_ylabel('Attendance')
axes[1].set_xlabel('Round')
axes[1].set_title(f'Quantum (q=0.4): σ²/N = {result_q.volatility:.3f}')
axes[1].set_ylim(20, 80)

plt.tight_layout()
plt.show()

In [ ]:
# Attendance histograms
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

params = MinorityGameParams(n_agents=101, memory=2, n_rounds=1000, seed=42)

for ax, q, color, label in zip(axes, [0.0, 0.4], ['blue', 'green'], ['Classical', 'Quantum (q=0.4)']):
    result = run_minority_game(params, quantumness=q)
    ax.hist(result.attendance, bins=30, density=True, color=color, alpha=0.7, edgecolor='white')
    ax.axvline(x=50.5, color='red', linestyle='--', label='N/2')
    ax.set_xlabel('Attendance')
    ax.set_ylabel('Density')
    ax.set_title(f'{label}: σ²/N = {result.volatility:.3f}')
    ax.legend()

plt.suptitle('Attendance Distribution: Classical vs Quantum', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## 5. Connection to the QCCCM Framework

The minority game connects to several QCCCM modules:

| Minority Game concept | QCCCM module | Connection |
|---|---|---|
| Agent strategy selection | `models/alf_bridge.py` | Quantum EFE for policy evaluation |
| Multi-agent dynamics | `networks/multi_agent.py` | Mean-field belief evolution |
| Optimal policy assignment | `annealing/qubo.py` | QUBO formulation for anti-coordination |
| Strategy interference | `circuits/interference.py` | Quantum interference circuits |
| Phase transition | `benchmarks/` | Scaling analysis |

The `quantumness` parameter in the minority game plays the same role as in quantum EFE: it controls the strength of coherence that enables interference between decision paths.

---

## Exercises

### Basic

**Exercise 4.1:** Run the minority game with N=501 agents and vary M from 1 to 10. Plot the phase diagram and identify α_c. How does the critical point change with N?

**Exercise 4.2:** Fix M=3 and vary the number of strategies S from 2 to 8. How does increasing the strategy pool affect volatility?

### Stretch

**Exercise 4.3 (Annealing):** Use `qcccm.annealing.solve_policy_assignment` to find the optimal anti-coordinated assignment of strategies to agents. Compare the annealer's solution to what agents learn through adaptation.

**Exercise 4.4 (Network minority game):** Connect agents via a `qcccm.networks.ring_graph` so each agent only sees their neighbors' choices (not the global minority). How does local information change the dynamics? Does quantum coherence help more or less with local information?

**Exercise 4.5 (Fit to data):** Generate choice data from the quantum minority game with known q, then use `qcccm.fitting.fit_mle` to recover the quantumness parameter. How many rounds of data do you need for accurate recovery?

## References

- Challet, D. & Zhang, Y.-C. (1997). Emergence of cooperation and organization in an evolutionary game. *Physica A*, 246, 407-418.
- Challet, D., Marsili, M. & Zhang, Y.-C. (2005). *Minority Games*. Oxford University Press.
- Moro, E. (2004). The Minority Game: An Introductory Guide. [Blog post]
- Busemeyer, J.R. & Bruza, P.D. (2012). *Quantum Models of Cognition and Decision*.